# Baseline lineal — regresión logística sobre conectividad estática

Implementa la contingencia de `docs/PLAN_RESPUESTA_REVISORES.md` §9.1 (activada por la
enmienda del 3 de agosto de 2026). Dieciséis corridas: cuatro sitios × los cuatro grupos de ROIs (12, 18, 39, 116).
El comparador cambia según el grupo — ver la celda 3.

Este notebook es deliberadamente distinto de `tdha_experimentos.ipynb`: no necesita GPU ni
Colab (una regresión logística sobre 66 o 6.670 columnas corre en CPU en segundos), no tiene
curva de entrenamiento que reportar, y su única decisión de diseño —la penalización L2,
`C=1.0`— está fija por la enmienda y no se edita aquí. El guardarraíl de partición
(`split_fingerprint`, `bold_hash`, `roi_indices_hash` contra la corrida BrainNetCNN pareada)
está embebido en `run_baseline_ml.py` y se ejecuta automáticamente antes de escribir cualquier
resultado: si no coincide, el script se detiene solo.


## 1. Preparar el entorno

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/jpospinalo/tdha-revision.git"

def _es_repo_valido(p):
    return (p / "src" / "run_baseline_ml.py").exists()

if "google.colab" in sys.modules or os.path.exists("/content"):
    # Colab: clonar (o actualizar) bajo /content, igual que tdha_experimentos.ipynb.
    ROOT = Path("/content") / REPO.rstrip("/").split("/")[-1].replace(".git", "")
    if ROOT.exists():
        subprocess.run(["git", "-C", str(ROOT), "pull"], check=False)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
else:
    # Entorno local: reutilizar el checkout desde el que se abrió este notebook,
    # sin clonar de nuevo. Busca hacia arriba desde el directorio actual.
    candidato = Path.cwd()
    ROOT = None
    for p in [candidato, *candidato.parents]:
        if _es_repo_valido(p):
            ROOT = p
            break
    if ROOT is None:
        raise RuntimeError(
            "No se encontró un checkout local de tdha-revision (src/run_baseline_ml.py "
            "no aparece en ningún directorio padre). Clone el repositorio manualmente o "
            "abra este notebook desde dentro de él."
        )

os.chdir(ROOT / "src")
sys.path.insert(0, ".")
print("directorio de trabajo:", os.getcwd())


In [ ]:
# Dependencias: numpy, pandas, scikit-learn, joblib. No requiere TensorFlow ni GPU.
import importlib
for pkg in ("numpy", "pandas", "sklearn", "joblib"):
    m = importlib.import_module(pkg)
    print(f"{pkg}: {getattr(m, '__version__', 'ok')}")


## 2. Configuración

**Esta es la única celda que hay que editar.** A diferencia de `tdha_experimentos.ipynb`, no
hay bloques de arquitectura, enventanado ni hiperparámetros de entrenamiento: la penalización
está fija por la enmienda de §9.1 y no es una decisión que se tome aquí. Lo único que se elige
es qué combinaciones de sitio y grupo de ROIs correr.


In [ ]:
SITIOS = ["NYU", "Peking", "NeuroIMAGE", "OHSU"]
ROI_SETS = ["12", "18", "39", "116"]

# Para una corrida de prueba, reduzca las listas de arriba antes de la primera ejecución
# formal. La campaña completa son las 4 x 4 = 16 combinaciones.
COMBINACIONES = [(s, r) for s in SITIOS for r in ROI_SETS]
print(f"{len(COMBINACIONES)} corridas configuradas:")
for s, r in COMBINACIONES:
    print(f"  {s:11s} roi_set={r}")


## 3. Comprobación previa: ¿existe el comparador pareado?

Hay dos familias de comparador, y `run_baseline_ml.py` las resuelve automáticamente:

- **roi_set=12:** compara contra la corrida BrainNetCNN `static` (§8.1 del plan). Un solo factor cambia — la arquitectura —, sin confusión de representación.
- **roi_set=18, 39, 116:** no existe corrida `static` para esos tamaños (crearla reabriría la campaña de diez corridas ya cerrada), así que el script busca en `run_manifest.csv` la corrida `ordered` que ya es la referencia primaria de la Tabla 6. Esa comparación **sí** confunde representación con arquitectura — mismo aviso que la dimensión «signal representation» de §2.6 del manuscrito — y queda marcada como tal en `config.json` (`representation_confound: true`).

Esta celda solo informa qué comparador usará cada combinación antes de correr nada.


In [ ]:
from pathlib import Path
import glob, csv

REPO_ROOT = Path("..").resolve()
manifest_path = REPO_ROOT / "analysis" / "roi_comparison" / "config" / "run_manifest.csv"
manifest = list(csv.DictReader(open(manifest_path))) if manifest_path.exists() else []

sin_comparador = []
for site, roi in COMBINACIONES:
    static_matches = glob.glob(f"../results/runs/{roi}/{site}_rois{roi}_static_brainnetcnn_*")
    if static_matches:
        print(f"  {site:11s} roi_set={roi:>4s}  comparador static (sin confusión)")
        continue
    manifest_matches = [r for r in manifest if r["site"] == site and r["roi_set"] == roi and r["include"].lower() == "true"]
    if manifest_matches:
        print(f"  {site:11s} roi_set={roi:>4s}  comparador ordered vía run_manifest.csv (CONFUSIÓN declarada)")
    else:
        print(f"  {site:11s} roi_set={roi:>4s}  SIN COMPARADOR — esta corrida se detendrá sola")
        sin_comparador.append((site, roi))

if sin_comparador:
    print(f"\n{len(sin_comparador)} combinaciones sin comparador; revise antes de continuar.")


## 4. La corrida

`run_baseline_ml.py` hace, por combinación: carga el BOLD, construye las mismas particiones que
la corrida BrainNetCNN pareada (misma semilla, mismo `n_splits`/`n_repeats`/`inner_val_frac`),
verifica el guardarraíl de tres hashes, calcula la conectividad estática, ajusta la regresión
logística pliegue por pliegue con estandarización solo sobre `fit`, y escribe
`config.json`/`folds.csv`/`predictions_val.csv`/`metrics_val.csv`/`metrics_train.csv`/`resumen.md`
en `results/runs/{roi_set}/{run_id}/`, igual que las corridas de BrainNetCNN.

Si el guardarraíl falla para una combinación, esa corrida se detiene con `SystemExit` y **no**
escribe ningún archivo; las demás combinaciones de la lista siguen normalmente.


In [ ]:
from run_baseline_ml import main as ejecutar_baseline

RUN_IDS = {}
for site, roi in COMBINACIONES:
    print(f"\n{'='*70}\n{site} — roi_set={roi}\n{'='*70}")
    try:
        run_id = ejecutar_baseline(["--site", site, "--roi-set", roi])
        RUN_IDS[(site, roi)] = run_id
    except SystemExit as exc:
        print(f"DETENIDA: {exc}")
        RUN_IDS[(site, roi)] = None

print("\nresumen:")
for (site, roi), run_id in RUN_IDS.items():
    print(f"  {site:11s} roi_set={roi:>4s}  ->  {run_id or 'NO EJECUTADA'}")


## 5. Resultados

In [ ]:
import json
from pathlib import Path

for (site, roi), run_id in RUN_IDS.items():
    if run_id is None:
        continue
    cfg = json.loads((Path("../results/runs") / roi / run_id / "config.json").read_text())
    print(f"{site:11s} roi_set={roi:>4s}  AUC OOF media={cfg['oof_auc_mean']:.4f}  "
          f"por repetición={['%.3f' % a for a in cfg['oof_auc_by_repeat']]}")


## 6. Validación ligera (no `compile_results.validate_run_artifacts`)

Ese validador exige `history.csv` con una serie de épocas completa por pliegue y los campos
`best_epoch`/`best_monitor_value` de `EarlyStopping`: son conceptos de entrenamiento con Keras
que una regresión logística no tiene. Reutilizarlo tal cual fallaría, no porque el baseline esté
mal, sino porque el validador comprueba algo que este modelo no produce. Esta celda verifica en
su lugar lo que sí aplica a cualquier corrida de este pipeline: cobertura de sujetos, ausencia de
solapamiento entre particiones y rango válido de las probabilidades.


In [ ]:
import pandas as pd

def validar_corrida_ligera(site, roi, run_id):
    ruta = Path("../results/runs") / roi / run_id
    folds = pd.read_csv(ruta / "folds.csv")
    preds = pd.read_csv(ruta / "predictions_val.csv")
    problemas = []

    for (fold, repeat), grupo in folds.groupby(["fold", "repeat"]):
        conjuntos = {s: set(grupo[grupo["split"] == s]["subject"]) for s in ("fit", "inner_val", "outer_val")}
        if conjuntos["fit"] & conjuntos["inner_val"] or conjuntos["fit"] & conjuntos["outer_val"] or conjuntos["inner_val"] & conjuntos["outer_val"]:
            problemas.append(f"fold {fold} repeat {repeat}: particiones solapadas")

    if not preds["y_prob"].between(0, 1).all():
        problemas.append("y_prob fuera de [0, 1]")
    if not preds["y_true"].isin([0, 1]).all():
        problemas.append("y_true fuera de {0, 1}")

    n_subjects_cfg = json.loads((ruta / "config.json").read_text())["n_subjects"]
    n_repeats = folds["repeat"].max()
    cobertura_esperada = n_subjects_cfg * n_repeats
    if len(preds) != cobertura_esperada:
        problemas.append(f"predicciones={len(preds)}, esperado={cobertura_esperada}")

    return problemas

for (site, roi), run_id in RUN_IDS.items():
    if run_id is None:
        continue
    problemas = validar_corrida_ligera(site, roi, run_id)
    estado = "OK" if not problemas else f"PROBLEMAS: {problemas}"
    print(f"  {site:11s} roi_set={roi:>4s}  {estado}")


## 7. Subir los resultados

Mismo flujo que `tdha_experimentos.ipynb` §7: necesita un token de acceso personal de GitHub con
permiso de escritura sobre este repositorio.


In [ ]:
import getpass

NOMBRE = ""   # completar antes de ejecutar
CORREO = ""   # completar antes de ejecutar

if not NOMBRE or not CORREO:
    raise ValueError("Complete NOMBRE y CORREO arriba y vuelva a ejecutar esta celda.")

TOKEN = getpass.getpass("Token de GitHub (no se mostrará en pantalla): ")


In [ ]:
import subprocess

def git(*args):
    r = subprocess.run(["git"] + list(args), cwd="..", capture_output=True, text=True)
    print((r.stdout + r.stderr).strip())
    return r.returncode

git("config", "user.name", NOMBRE)
git("config", "user.email", CORREO)
git("add", "results/runs/")
git("commit", "-m", f"Baseline logreg (§9.1, enmienda 3 ago 2026): {list(RUN_IDS.values())}")
# git("push", f"https://{TOKEN}@github.com/jpospinalo/tdha-revision.git", "main")
